# Three-Model Ablation Study

This notebook conducts a clean ablation study comparing:
- **Model A (Baseline)**: popularity, weight, market, year only
- **Model B (ENAO Similarity)**: baseline + ENAO co-occurrence weights
- **Model C (Hyperbolic Distance)**: baseline + hyperbolic distance

All models use identical Gradient Boosting hyperparameters.

## 1. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load dataset (make sure model_dataset2.csv is in the same folder as this notebook)
DATASET = 'model_dataset2.csv'
TARGET = 'log_streams'

df = pd.read_csv(DATASET)
print(f"✓ Dataset loaded: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df = df.dropna(subset=['log_ranking_score_source', 'log_ranking_score_target'])

# Encode market
le_market = LabelEncoder()
df['market_enc'] = le_market.fit_transform(df['market'])

# Check for required columns
required_cols = ['distance', 'enao_similarity', 'log_ranking_score_source', 
                 'log_ranking_score_target', 'log_weight', 'market_enc', 'year']

missing = [col for col in required_cols if col not in df.columns]
if missing:
    print(f"ERROR: Missing columns: {missing}")
    print("Please run build_dataset.ipynb first to generate enao_similarity column.")
else:
    print("✓ All required columns present")
    print(f"Dataset size: {len(df):,} collaborations")

## 2. Define Feature Sets for Each Model

In [ ]:
# Baseline features (no genre relationship)
features_baseline = [
    'log_ranking_score_source',
    'log_ranking_score_target',
    'log_weight',
    'market_enc',
    'year'
]

# Model A: Baseline only
features_A = features_baseline

# Model B: Baseline + ENAO similarity
features_B = features_baseline + ['enao_similarity']

# Model C: Baseline + Hyperbolic distance
features_C = features_baseline + ['distance']

print("Model A (Baseline):")
print(f"  Features: {features_A}")
print()
print("Model B (ENAO Similarity):")
print(f"  Features: {features_B}")
print()
print("Model C (Hyperbolic Distance):")
print(f"  Features: {features_C}")

## 3. Train/Test Split

Use the same split for all three models (stratified by market)

In [ ]:
# Target variable
y = df[TARGET]

# Train/test split (stratified by market, same for all models)
# Use random_state=42 to ensure reproducibility
X_full = df[features_C]  # Use all possible features for splitting
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['market']
)

# Extract feature subsets for each model
X_train_A = X_train_full[features_A]
X_test_A = X_test_full[features_A]

X_train_B = X_train_full[features_B]
X_test_B = X_test_full[features_B]

X_train_C = X_train_full[features_C]
X_test_C = X_test_full[features_C]

print(f"Train set: {len(X_train_A):,} samples")
print(f"Test set:  {len(X_test_A):,} samples")

## 4. Train All Three Models

Use identical hyperparameters (from the original prediction_model.ipynb that gave 0.684)

In [ ]:
# Hyperparameters (these gave Test R² = 0.684 in original notebook)
best_params = {
    'n_estimators': 300,
    'max_depth': 7,
    'learning_rate': 0.1,
    'subsample': 0.9,
    'min_samples_split': 5,
    'random_state': 42
}

print("Training three models with identical hyperparameters...")
print(f"Parameters: {best_params}")
print()

# ============================================================================
# Model A: Baseline
# ============================================================================
print("[1/3] Training Model A (Baseline)...")
model_A = GradientBoostingRegressor(**best_params)
model_A.fit(X_train_A, y_train)

# 5-fold CV on training set
cv_scores_A = cross_val_score(model_A, X_train_A, y_train, cv=5, scoring='r2')
# Test set performance
r2_test_A = model_A.score(X_test_A, y_test)

print(f"  CV R²:   {cv_scores_A.mean():.3f} (±{cv_scores_A.std():.3f})")
print(f"  Test R²: {r2_test_A:.3f}")
print()

# ============================================================================
# Model B: ENAO Similarity
# ============================================================================
print("[2/3] Training Model B (ENAO Similarity)...")
model_B = GradientBoostingRegressor(**best_params)
model_B.fit(X_train_B, y_train)

cv_scores_B = cross_val_score(model_B, X_train_B, y_train, cv=5, scoring='r2')
r2_test_B = model_B.score(X_test_B, y_test)

print(f"  CV R²:   {cv_scores_B.mean():.3f} (±{cv_scores_B.std():.3f})")
print(f"  Test R²: {r2_test_B:.3f}")
print()

# ============================================================================
# Model C: Hyperbolic Distance
# ============================================================================
print("[3/3] Training Model C (Hyperbolic Distance)...")
model_C = GradientBoostingRegressor(**best_params)
model_C.fit(X_train_C, y_train)

cv_scores_C = cross_val_score(model_C, X_train_C, y_train, cv=5, scoring='r2')
r2_test_C = model_C.score(X_test_C, y_test)

print(f"  CV R²:   {cv_scores_C.mean():.3f} (±{cv_scores_C.std():.3f})")
print(f"  Test R²: {r2_test_C:.3f}")
print()
print("✓ All models trained successfully")

## 5. Statistical Significance Testing

In [ ]:
# Get predictions for all models on test set
pred_A = model_A.predict(X_test_A)
pred_B = model_B.predict(X_test_B)
pred_C = model_C.predict(X_test_C)

# Compute squared errors
errors_A = (y_test.values - pred_A) ** 2
errors_B = (y_test.values - pred_B) ** 2
errors_C = (y_test.values - pred_C) ** 2

# Paired t-tests
t_BA, p_BA = scipy_stats.ttest_rel(errors_B, errors_A)  # ENAO vs Baseline
t_CA, p_CA = scipy_stats.ttest_rel(errors_C, errors_A)  # Hyperbolic vs Baseline
t_CB, p_CB = scipy_stats.ttest_rel(errors_C, errors_B)  # Hyperbolic vs ENAO

print("="*80)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*80)
print()
print("Paired t-tests (comparing squared errors):")
print()
print(f"  B vs A (ENAO vs Baseline):")
print(f"    t = {t_BA:.4f}, p = {p_BA:.6f} {'✓ significant' if p_BA < 0.05 else '✗ not significant'}")
print()
print(f"  C vs A (Hyperbolic vs Baseline):")
print(f"    t = {t_CA:.4f}, p = {p_CA:.6f} {'✓ significant' if p_CA < 0.05 else '✗ not significant'}")
print()
print(f"  C vs B (Hyperbolic vs ENAO):")
print(f"    t = {t_CB:.4f}, p = {p_CB:.6f} {'✓ significant' if p_CB < 0.05 else '✗ not significant'}")
print()
print("="*80)

## 6. Feature Importance Analysis

In [ ]:
print("="*80)
print("FEATURE IMPORTANCE")
print("="*80)
print()

# Model B: ENAO Similarity
print("Model B (ENAO Similarity):")
for feat, imp in zip(features_B, model_B.feature_importances_):
    print(f"  {feat:<30} {imp:.3f}")
print()

# Model C: Hyperbolic Distance
print("Model C (Hyperbolic Distance):")
for feat, imp in zip(features_C, model_C.feature_importances_):
    print(f"  {feat:<30} {imp:.3f}")
print()

# Compare the genre relationship features
enao_importance = model_B.feature_importances_[-1]  # Last feature
distance_importance = model_C.feature_importances_[0]  # First feature (distance)

print("Key comparison:")
print(f"  ENAO Similarity importance:     {enao_importance:.3f} ({enao_importance*100:.1f}%)")
print(f"  Hyperbolic Distance importance: {distance_importance:.3f} ({distance_importance*100:.1f}%)")
print(f"  Ratio (Distance / ENAO):        {distance_importance / enao_importance:.1f}x")
print()
print("="*80)

## 7. Summary Table

In [ ]:
print("="*80)
print("ABLATION STUDY RESULTS")
print("="*80)
print()
print(f"{'Model':<30} {'CV R²':<20} {'Test R²':<15} {'Improvement'}")
print("-" * 80)
print(f"{'A (Baseline)':<30} {cv_scores_A.mean():.3f} (±{cv_scores_A.std():.3f}){'':<6} {r2_test_A:.3f}{'':<10} {'—'}")
print(f"{'B (ENAO Similarity)':<30} {cv_scores_B.mean():.3f} (±{cv_scores_B.std():.3f}){'':<6} {r2_test_B:.3f}{'':<10} {'+' if r2_test_B >= r2_test_A else ''}{r2_test_B - r2_test_A:.3f}")
print(f"{'C (Hyperbolic Distance)':<30} {cv_scores_C.mean():.3f} (±{cv_scores_C.std():.3f}){'':<6} {r2_test_C:.3f}{'':<10} +{r2_test_C - r2_test_A:.3f}")
print()
print("Statistical Significance (α = 0.05):")
print(f"  B vs A (ENAO vs Baseline):        p = {p_BA:.6f} {'✓ reject H₀' if p_BA < 0.05 else '✗ fail to reject H₀'}")
print(f"  C vs A (Hyperbolic vs Baseline):  p = {p_CA:.6f} {'✓ reject H₀' if p_CA < 0.05 else '✗ fail to reject H₀'}")
print(f"  C vs B (Hyperbolic vs ENAO):      p = {p_CB:.6f} {'✓ reject H₀' if p_CB < 0.05 else '✗ fail to reject H₀'}")
print()
print("Feature Importance (genre relationship features):")
print(f"  ENAO Similarity:     {enao_importance:.3f} ({enao_importance*100:.1f}%)")
print(f"  Hyperbolic Distance: {distance_importance:.3f} ({distance_importance*100:.1f}%)")
print()
print("="*80)

# Interpretation
print()
print("KEY FINDINGS:")
print()
if r2_test_B <= r2_test_A + 0.001:
    print("1. ENAO similarity provides NO significant improvement over baseline")
else:
    print(f"1. ENAO similarity improves Test R² by {r2_test_B - r2_test_A:.3f}")

if p_CA < 0.001:
    print(f"2. Hyperbolic distance significantly improves Test R² by {r2_test_C - r2_test_A:.3f} (p < 0.001)")
    print(f"   - This is a {((r2_test_C - r2_test_A) / r2_test_A * 100):.1f}% relative improvement")
else:
    print(f"2. Hyperbolic distance improves Test R² by {r2_test_C - r2_test_A:.3f} (p = {p_CA:.3f})")

if p_CB < 0.001:
    print(f"3. Hyperbolic distance significantly outperforms ENAO similarity (p < 0.001)")
    print(f"   - Improvement: {r2_test_C - r2_test_B:.3f} in Test R²")

print(f"4. Hyperbolic distance is {distance_importance / enao_importance:.1f}x more important than ENAO similarity")
print()
print("CONCLUSION: The predictive value comes from the geometric structure learned")
print("by hyperbolic embeddings, not from the raw co-occurrence patterns.")